In [1]:
%pip install faiss-cpu
%pip install numpy
%pip install sentence-transformers
%pip install openai
%pip install python-dotenv



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import openai
load_dotenv()
client = openai.OpenAI()  # will automatically pick up from env


/home/lit2vec/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import json

def load_json_files(folder_path):
    data = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                try:
                    data.append(json.load(f))
                except Exception as e:
                    print(f"Error parsing {filename}: {e}")
    return data


In [4]:
records = load_json_files("./../../example_docs/lit2vec_1K_sample_records")

docs = []
for rec in records:
    corpus_id = rec.get("corpus_id")
    metadata = rec.get("metadata", {})
    paragraphs = rec.get("paragraphs", {})
    embeddings = rec.get("embeddings", {})

    for para_id, text in paragraphs.items():
        embedding = embeddings.get(para_id)
        if embedding:
            docs.append({
                "corpus_id": corpus_id,
                "paragraph_id": para_id,  
                "text": text,
                "embedding": embedding,
                "metadata": {
                    "corpus_id": corpus_id,
                    "paragraph_id": para_id,
                    "title": metadata.get("title"),
                    "authors": metadata.get("authors"),
                    "year": metadata.get("year"),
                    "doi": metadata.get("externalids", {}).get("DOI"),
                    "fields": metadata.get("s2fieldsofstudy"),
                    "subfield_preds": rec.get("predicted_subfield", {})
                }
            })


In [5]:
from sentence_transformers import util
import faiss
import numpy as np

# Convert embeddings to numpy
dim = len(docs[0]['embedding'])  # e.g., 768
index = faiss.IndexFlatL2(dim)
vectors = np.array([doc['embedding'] for doc in docs]).astype("float32")

# Add to FAISS
index.add(vectors)

# Map vectors to texts/metadata
doc_texts = [doc['text'] for doc in docs]
doc_meta = [doc['metadata'] for doc in docs]


In [6]:
from sentence_transformers import SentenceTransformer

retriever_model = SentenceTransformer("intfloat/e5-large-v2")

def retrieve(query, top_k=5):
    query_emb = retriever_model.encode(query, normalize_embeddings=True)
    D, I = index.search(np.array([query_emb]), top_k)
    
    results = []
    for i in I[0]:
        doc = docs[i]
        results.append((
            doc['text'],
            {
                "paragraph_id": doc['paragraph_id'],# You need to store this during preprocessing
                "corpus_id": doc['corpus_id'],
                "title": doc['metadata']['title'],
                "year": doc['metadata']['year'],
                "doi": doc['metadata']['doi'],
                "authors": doc['metadata']['authors'],
            }
        ))
    return results



In [7]:
import openai

# Ensure you have set your OpenAI key:
# openai.api_key = "your-api-key" or use environment variable

client = openai.OpenAI()

def generate_answer_with_inline_refs(query, retrieved_docs):
    # Build prompt with paragraph identifiers
    context_blocks = []
    reference_data = {}

    for idx, (text, meta) in enumerate(retrieved_docs):
        para_id = meta.get("paragraph_id", f"unknownP{idx}")
        corpus_id = meta.get("corpus_id", "unknown")
        context_blocks.append(f"[{para_id}]\n{text}")
        
        # Store doc-level metadata for citation
        if corpus_id not in reference_data:
            reference_data[corpus_id] = {
                "corpus_id": corpus_id,
                "title": meta.get("title", "Unknown Title"),
                "year": meta.get("year", "n.d."),
                "doi": meta.get("doi", None),
                "authors": meta.get("authors", []),
            }

    context = "\n\n".join(context_blocks)

    # Build prompt
    prompt = f"""You are a scientific assistant. Use the context passages below to answer the question.

Cite each specific supporting statement using its paragraph ID in parentheses (e.g., (37254803P1)). Write the answer as a coherent paragraph, not as a list. Avoid bullet points.

Context:
{context}

Question:
{query}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    answer = response.choices[0].message.content.strip()

    # Build References section
    references = "\n\nReferences:\n"
    for corpus_id, meta in reference_data.items():
        authors = ", ".join(a.get("name", "") for a in meta["authors"]) or "Unknown Authors"
        title = meta["title"]
        year = meta["year"]
        doi = meta["doi"]
        doi_part = f" https://doi.org/{doi}" if doi else ""
        references += f"[{corpus_id}] {authors} ({year}). *{title}*.{doi_part}\n"

    return answer + references



In [8]:
query = "What are the benefits of EGCG in oxidative stress?"
retrieved = retrieve(query)
response = generate_answer_with_inline_refs(query, retrieved)
print(response)


EGCG, a polyphenolic compound found in green tea, is recognized as an effective antioxidant due to its abundant presence and distinct chemical structure, which includes two triphenolic groups (9587808P6). Its antioxidant properties are linked to the blockade of nNOS and iNOS induction, and the scavenging of free radicals, which can reduce lipid peroxidation, suggesting a neuroprotective potential (9587808P7). In the context of oxidative stress induced by peripheral nerve injury, EGCG has shown the ability to decrease NADPH-d/nNOS reactivity and cell death in motor neurons, indicating enhanced survival of these neurons (9587808P2). The protective effects of EGCG against neuronal damage are dose-dependent, and higher doses have been linked to the suppression of PNCI-induced NADPH-d/nNOS expression in brainstem motor neurons, attributed to its potent antioxidant capacity (9587808P32). Additionally, EGCG has demonstrated protective effects against neuronal damage and brain edema following 